# Natural Language Understanding

Ziele der heutigen Übung:
- Erstellung eines Datensatzes an Nutzeräußerungen mit Hilfe eines LLMs
- Intent Klassifikation regelbasiert
- Implementierung einer NLU auf Basis der Library stanza

## Erstellung Datensatz

Wir möchten in der Übung einen Datensatz nutzen, welcher Nutzeräußerungen zu zwei Intents enthält:
1) ```Pizza_Bestellen```: Ein Nutzer soll mit einem Sprachdialogsystem eine Bestellung für Pizza aufgeben können
2) ```Oeffnungszeiten_Erfragen```: Ein Nutzer soll mit einem Sprachdialogsystem erfragen können, ob die Pizzeria geöffnet hat.

Weiterhin sollen zu jedem Intent folgende Entities möglich sein:
1) ```Pizza_Bestellen```: Belag, Größe, Extra_Käse
2) ```Oeffnungszeiten_Erfragen```: Wochentag

Es geht nichts über reale Dialoge, aber zur ersten Implmenetierung hilft uns ein LLM ungemein, indem es mit einem einfachen Prompt einen entsprechenden synthetischen Datensatz generieren kann.

Gehe mit einem LLM deiner Wahl folgendermaßen vor:
1) Definiere zuerst die Persona des Sprechers, d.h. wer soll mit unserem Sprachdialogsystem interagieren. 
2) Erstelle einen Prompt mit der Definition des Sprechers sowie mit der Spezifikation für Intents/Entities (s.o.)
3) Füge das gewünschte Ausgabeformat dem Prompt hinzu: 200 Nutzeräußerungen als .CSV-Datei mit folgenden Spalten: "utterance" als String, "intent" als String, "entities" als JSON


Wichtige Einschränkung: der Einfachheit halber betrachten wir hier nur den Happy-Path, d.h. wir haben keine Äußerungen die zu keinem der beiden Intents gehören und keine Äußerungen mit Fehlern.


In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 400)

CSV_FILE = ''  #TODO: generiertes CSV File hier referenzieren
df = pd.read_csv(CSV_FILE, sep=";", skiprows=1)
df.head()

Um im folgenden einfacher mit dem Datensatz arbeiten zu können, erstellen wir für jeden Intent einen extra dataframe:

In [ ]:
df_order = df[df['intent'] == 'Pizza_Bestellen']
df_opening_hours = df[df['intent'] == 'Oeffnungszeiten_Erfragen']                                                               

Anschließend teilen wir die Daten in einen 20:80 train-test Split (! genau andersrum als beim klassischen Machine Learning!). Wir schauen uns die 20 Trainingsdaten an und versuchen daraus unsere Regeln für die NLU abzuleiten. Im Testset testen wir, wie gut diese funktionieren.

In [ ]:
df_order_train = None  # TODO: split 0-19
df_order_test = None  # TODO: split >20
df_opening_hours_train = None  # TODO: split 0-19
df_opening_hours_test = None  # TODO: split >20

In [ ]:
df_order_train

In [ ]:
df_opening_hours_train

## NLU mit einfachen Regeln

In [ ]:
# Regular expression may help us here, however, it also works with string comparision
import re

Wir schauen uns das Trainingsdatenset genau an und versuchen Wörter zu identifizieren, die auf den einen oder anderen Intent hinweisen:

In [ ]:
df_order_train['utterance']

Im Folgenden erstellen wir eine Maske, welche für jede Zeile in unserem Test-Datensatz den Intent abhängig unserer identifizierten Wörter für ein Intent findet. Gebe danach aus wie viele Äußerungen korrekt klassifiziert wurden:

In [ ]:
intent_keywords_order = []  # TODO: keywords die auf ein Intent hinweisen.

regex = '|'.join(intent_keywords_order)
mask = df_order_test['utterance'].str.contains(regex, na=False, flags=re.IGNORECASE, regex=True)

print(f"Korrekt klassifizierte Intents: {int(mask.value_counts().values[0])} von {len(df_order_test)}")

Dasselbe können wir nun noch für das Intent ```Oeffnungszeiten_Erfragen```machen:

In [ ]:
df_opening_hours_train['utterance']

In [ ]:
# TODO: siehe Source Code für df_order_test

print(f"Korrekt klassifizierte Intents: {int(mask.value_counts().values[0])} von {len(df_order_test)}")

Brainstorme mit deinem Nachbarn, wie man die Erkennung von Entities implementieren könnte. Gehe auf Entities ein, die nicht eindeutig mit einem Wort abbildbar sind, wie zum Beispiel:
```Ich bestelle eine große Pizza Salami mit Käse``` vs. ```Ich bestelle eine extra große Pizza Salami ohne Käse```

## NLU mit Stanza

Bessere Ergebnisse erzielen wir, wenn wir die Äußerung mit NLP-Verfahren analysieren lassen und aus diesen Ergebnissen versuchen unsere Intents und Entities zu identifizieren. Wir möchten hierfür die Bibliothek https://stanfordnlp.github.io/stanza/ einsetzen. Installiere sie mit pip:

In [ ]:
pip install stanza

Anschließend initialisieren wir die Pipeline mit folgenden ```processors='tokenize, pos, lemma, depparse, ner'```. Zusätzlich benötigt sie noch die Sprache in welcher unsere Nutzeräußerungen sind.

Anschließend lassen wir die beiden Äußerungen ```Ich bestelle eine große Pizza Salami mit Käse``` und ```Ich bestelle eine extra große Pizza Salami ohne Käse``` durch die Pipeline analysieren und ausgeben.

Speziell hilfreich für die Interpretation sind folgende Werte:
- lemma
- id und head
- upos oder xpos

Eine Erklärung dazu findest du in der Doku: https://stanfordnlp.github.io/stanza/data_objects.html 

In [ ]:
import stanza

nlp = None  # TODO: stanza Pipeline initialisieren

order_with_cheese = nlp('Ich bestelle eine große Pizza Salami mit Käse').to_dict()
order_without_cheese = nlp('Ich bestelle eine große Pizza Salami ohne Käse').to_dict()

display(order_with_cheese)
display(order_without_cheese)

Nun machen wir das für alle Äußerungen und speichern die Stanza Ergebnisse in die Dataframes:

In [ ]:
df_opening_hours_train['nlu'] = df_opening_hours_train.apply(lambda row: nlp(row['utterance']).to_dict(), axis=1)
df_opening_hours_test['nlu'] = df_opening_hours_test.apply(lambda row: nlp(row['utterance']).to_dict(), axis=1)
df_order_train['nlu'] = df_order_train.apply(lambda row: nlp(row['utterance']).to_dict(), axis=1)
df_order_test['nlu'] = df_order_test.apply(lambda row: nlp(row['utterance']).to_dict(), axis=1)

Einzelne NLU-Ergebnisse können wir uns folgendermaßen anschauen:

In [ ]:
index = 2

display(df_order_train['nlu'].loc[[index]].values[0])

Nehme dir nun verschiedene Sätze aus dem Trainingsdatensatz und baue eine eigene NLU, die Intents und Entities erkennen kann: 

In [ ]:
def nlu(stanza_result: dict) -> tuple[str, dict]:
    ret_intent = None
    ret_entities = {}
    
    # TODO: eigene NLU implementieren. stanza_result ist jeweils eine Interpretation einer Nutzeräußerung. 
    # mit for word in stanza_result[0]: iterieren wir über die Wörter einer Interpretation.

    return (ret_intent, ret_entities)

mask_opening_hours = df_opening_hours_test['nlu'].apply(nlu)
mask_orders = df_order_test['nlu'].apply(nlu)

mask_opening_hours
mask_orders

Eine vollständige NLU mit Regeln zu erstellen ist nicht ganz einfach. Im Jupyter Notebook der Lösung ist ein mit Github Copilot generiertes Skript.